# Lab - Classification

In this lab, we are going to build a classification module. When given an image of a handwritten digit like the one below, the model will be able to tell which digit is in the image.

<img src='test2.jpg'>

In [27]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier  # MLP is an NN
from sklearn import svm
import numpy as np
import argparse
import imutils  # If you are unable to install this library, ask the TA; we only need this in extract_hsv_histogram.
import cv2
import os
import random


# Depending on library versions on your system, one of the following imports 
from sklearn.model_selection import train_test_split
#from sklearn.cross_validation import train_test_split

In [28]:
path_to_dataset = r'digits_dataset'
target_img_size = (32, 32) # fix image size because classification algorithms THAT WE WILL USE HERE expect that

# We are going to fix the random seed to make our experiments reproducible 
# since some algorithms use pseudorandom generators
random_seed = 42  
random.seed(random_seed)
np.random.seed(random_seed)

## Part I - Feature Extraction

In this part, we are going to implement three functions. Each one will extract a different set of features from the image. The three sets are:

1. Histogram of the pixel values features (this is the histogram you know, but on the HSV channels)
2. Histogram of Gradients (HoG) features
3. Raw pixels (basically, not doing any feature extraction and just supplying the input image to the classifier)

In [29]:
def extract_hsv_histogram(img):
    """
    TODO
    1. Resize the image to target_img_size using cv2.resize
    2. Convert the image from BGR representation (cv2 is BGR not RGB) to HSV using cv2.cvtColor
    3. Acquire the histogram using the cv2.calcHist. Apply the functions on the 3 channels. For the bins 
        parameter pass (8, 8, 8). For the ranges parameter pass ([0, 180, 0, 256, 0, 256]). Name the histogram
        <hist>.
    """
    
    img = cv2.resize(img, target_img_size)
    hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv_img], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    
    if imutils.is_cv2():
        hist = cv2.normalize(hist)
    else:
        cv2.normalize(hist, hist)
    return hist.flatten()     

In [30]:
def extract_hog_features(img):
    """
    TODO
    You won't implement anything in this function. You just need to understand it 
    and understand its parameters (i.e win_size, cell_size, ... etc)
    """
    img = cv2.resize(img, target_img_size)
    win_size = (32, 32)
    cell_size = (4, 4)
    block_size_in_cells = (2, 2)
    
    block_size = (block_size_in_cells[1] * cell_size[1], block_size_in_cells[0] * cell_size[0])
    block_stride = (cell_size[1], cell_size[0])
    nbins = 9  # Number of orientation bins
    hog = cv2.HOGDescriptor(win_size, block_size, block_stride, cell_size, nbins)
    h = hog.compute(img)
    h = h.flatten()
    return h.flatten()

In [31]:
def extract_raw_pixels(img):
    """
    TODO
    The classification algorithms we are going to use expect the input to be a vector not a matrix. 
    This is because they are general purpose and don't work only on images.
    CNNs, on the other hand, expect matrices since they operate on images and exploit the 
    arrangement of pixels in the 2-D space.
    
    So, what we only need to do in this function is to resize and flatten the image.
    """
    return cv2.resize(img, target_img_size).flatten()

In [32]:
def extract_features(img, feature_set='hog'):
    """
    TODO
    Given either 'hsv_hist', 'hog', 'raw', call the respective function and return its output
    """
    if feature_set == 'hsv_hist':
        return extract_hsv_histogram(img)
    elif feature_set == 'hog':
        return extract_hog_features(img)
    elif feature_set == 'raw':
        return extract_raw_pixels(img)

The following function will extract the features and the label of each image in our dataset and save it in RAM. We normally don't save datasets in RAM, but this dataset is small.

In [33]:
def load_dataset(feature_set='hog'):
    features = []
    labels = []
    img_filenames = os.listdir(path_to_dataset)

    for i, fn in enumerate(img_filenames):
        if fn.split('.')[-1] != 'jpg':
            continue

        label = fn.split('.')[0]
        labels.append(label)

        path = os.path.join(path_to_dataset, fn)
        img = cv2.imread(path)
        features.append(extract_features(img, feature_set))
        
        # show an update every 1,000 images
        if i > 0 and i % 1000 == 0:
            print("[INFO] processed {}/{}".format(i, len(img_filenames)))
        
    return features, labels        

## Part II - Classification

In this part, we will test the classification performance of SVM, KNN, & NNs given our features.

In [34]:
# TODO understand the hyperparameters of each classifier
classifiers = {
    'SVM': svm.LinearSVC(random_state=random_seed),
    'SVM2': svm.LinearSVC(random_state=random_seed, C=5),
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'KNN2': KNeighborsClassifier(n_neighbors=5, weights="distance"),
    'NN': MLPClassifier(solver='sgd', random_state=random_seed, hidden_layer_sizes=(500,), max_iter=20, verbose=1),
    'NN2': MLPClassifier(solver='adam', random_state=random_seed, hidden_layer_sizes=(300,200), max_iter=15, verbose=1)
}

In [35]:
# This function will test all our classifiers on a specific feature set
def run_experiment(feature_set):
    
    # Load dataset with extracted features
    print('Loading dataset. This will take time ...')
    features, labels = load_dataset(feature_set)
    print('Finished loading dataset.')
    
    # Since we don't want to know the performance of our classifier on images it has seen before
    # we are going to withhold some images that we will test the classifier on after training 
    train_features, test_features, train_labels, test_labels = train_test_split(
        features, labels, test_size=0.2, random_state=random_seed)
    
    for model_name, model in classifiers.items():
        print('############## Training', model_name, "##############")
        # Train the model only on the training features
        model.fit(train_features, train_labels)
        
        # Test the model on images it hasn't seen before
        accuracy = model.score(test_features, test_labels)
        
        print(model_name, 'accuracy:', accuracy*100, '%')

Now, we see how each classifier and each feature set performs

In [36]:
run_experiment('hog')
"""
You should get the following test accuracies the first time 

SVM accuracy ~ 97.70833333333333
KNN accuracy ~ 96.52777777777779
NN accuracy ~ 93.95833333333333
"""

Loading dataset. This will take time ...
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
Finished loading dataset.
############## Training SVM ##############
SVM accuracy: 97.70833333333333 %
############## Training SVM2 ##############
SVM2 accuracy: 97.5 %
############## Training KNN ##############
KNN accuracy: 97.36111111111111 %
############## Training KNN2 ##############
KNN2 accuracy: 97.43055555555556 %
############## Training NN ##############
Iteration 1, loss = 2.15756547
Iteration 2, loss = 1.99567094
Iteration 3, loss = 1.83553434
Iteration 4, loss = 1.68166403
Iteration 5, loss = 1.53398460
Iteration 6, loss = 1.39403757
Iteration 7, loss = 1.26457832
Iteration 8, loss = 1.14758132
Iteration 9, loss = 1.04321463
Iteration 10, loss = 0.95153998
Iteration 11, loss = 0.87123011
Iteration 12, loss = 0.80120292
Iteration 13, loss = 0.74021

/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 1, loss = 0.67714776
Iteration 2, loss = 0.10985984
Iteration 3, loss = 0.07153568
Iteration 4, loss = 0.04873447
Iteration 5, loss = 0.03941546
Iteration 6, loss = 0.02476629
Iteration 7, loss = 0.02003512
Iteration 8, loss = 0.01266459
Iteration 9, loss = 0.01032499
Iteration 10, loss = 0.00740802
Iteration 11, loss = 0.00659599
Iteration 12, loss = 0.00385086
Iteration 13, loss = 0.00303348
Iteration 14, loss = 0.00276967
Iteration 15, loss = 0.00222752
NN2 accuracy: 97.63888888888889 %


/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


'\nYou should get the following test accuracies the first time \n\nSVM accuracy ~ 97.70833333333333\nKNN accuracy ~ 96.52777777777779\nNN accuracy ~ 93.95833333333333\n'

In [37]:
run_experiment('hsv_hist')
"""
You should get the following test accuracies the first time 

SVM accuracy ~ 32.083333333333336
KNN accuracy ~ 32.708333333333336
NN accuracy ~ 9.722222222222223
"""

# Why low accuracies?

Loading dataset. This will take time ...
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
Finished loading dataset.
############## Training SVM ##############
SVM accuracy: 31.11111111111111 %
############## Training SVM2 ##############
SVM2 accuracy: 36.45833333333333 %
############## Training KNN ##############
KNN accuracy: 29.86111111111111 %
############## Training KNN2 ##############
KNN2 accuracy: 31.041666666666668 %
############## Training NN ##############
Iteration 1, loss = 2.20356627
Iteration 2, loss = 2.20221406
Iteration 3, loss = 2.20105609
Iteration 4, loss = 2.20008368
Iteration 5, loss = 2.19941320
Iteration 6, loss = 2.19882558
Iteration 7, loss = 2.19834381
Iteration 8, loss = 2.19799954
Iteration 9, loss = 2.19766822
Iteration 10, loss = 2.19743026
Iteration 11, loss = 2.19720650
Iteration 12, loss = 2.19705779
Iteration 13, 

/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 2, loss = 2.18881438
Iteration 3, loss = 2.16096876
Iteration 4, loss = 2.09235675
Iteration 5, loss = 2.00397351
Iteration 6, loss = 1.92582183
Iteration 7, loss = 1.86825400
Iteration 8, loss = 1.82522155
Iteration 9, loss = 1.79486230
Iteration 10, loss = 1.77623307
Iteration 11, loss = 1.76678348
Iteration 12, loss = 1.76013358
Iteration 13, loss = 1.75336681
Iteration 14, loss = 1.74840151
Iteration 15, loss = 1.74530292
NN2 accuracy: 33.263888888888886 %


/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


'\nYou should get the following test accuracies the first time \n\nSVM accuracy ~ 32.083333333333336\nKNN accuracy ~ 32.708333333333336\nNN accuracy ~ 9.722222222222223\n'

In [38]:
run_experiment('raw')
"""
You should get the following test accuracies the first time 

SVM accuracy ~ 85.06944444444444
KNN accuracy ~ 93.95833333333333
NN accuracy ~ 88.68055555555556
"""

Loading dataset. This will take time ...
[INFO] processed 1000/7200
[INFO] processed 2000/7200
[INFO] processed 3000/7200
[INFO] processed 4000/7200
[INFO] processed 5000/7200
[INFO] processed 6000/7200
[INFO] processed 7000/7200
Finished loading dataset.
############## Training SVM ##############


/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM accuracy: 82.43055555555556 %
############## Training SVM2 ##############


/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


SVM2 accuracy: 82.56944444444444 %
############## Training KNN ##############
KNN accuracy: 94.02777777777777 %
############## Training KNN2 ##############
KNN2 accuracy: 95.0 %
############## Training NN ##############
Iteration 1, loss = 10.51227718
Iteration 2, loss = 1.35391043
Iteration 3, loss = 0.94708846
Iteration 4, loss = 0.74204435
Iteration 5, loss = 0.65452712
Iteration 6, loss = 0.55007074
Iteration 7, loss = 0.51946275
Iteration 8, loss = 0.40648945
Iteration 9, loss = 0.37653594
Iteration 10, loss = 0.29577068
Iteration 11, loss = 0.27326118
Iteration 12, loss = 0.25733033
Iteration 13, loss = 0.22796591
Iteration 14, loss = 0.21732591
Iteration 15, loss = 0.20410016
Iteration 16, loss = 0.19166321
Iteration 17, loss = 0.19354126
Iteration 18, loss = 0.18596876
Iteration 19, loss = 0.17369610
Iteration 20, loss = 0.16672983
NN accuracy: 90.13888888888889 %
############## Training NN2 ##############


/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 1, loss = 9.80110381
Iteration 2, loss = 2.30181036
Iteration 3, loss = 1.44285972
Iteration 4, loss = 1.14435655
Iteration 5, loss = 0.79078304
Iteration 6, loss = 0.52027618
Iteration 7, loss = 0.63032925
Iteration 8, loss = 0.42025902
Iteration 9, loss = 0.25796215
Iteration 10, loss = 0.35070472
Iteration 11, loss = 0.31868137
Iteration 12, loss = 0.28671642
Iteration 13, loss = 0.12986517
Iteration 14, loss = 0.13527823
Iteration 15, loss = 0.14768718
NN2 accuracy: 90.97222222222221 %


/media/mohamed/F4B0F1C1B0F18A7E/DEPI Data Science/Machine Learning/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


'\nYou should get the following test accuracies the first time \n\nSVM accuracy ~ 85.06944444444444\nKNN accuracy ~ 93.95833333333333\nNN accuracy ~ 88.68055555555556\n'

The classifiers list now has models trained on the last feature set you ran an experiment on. You can play around with it checking the probability it gives to each label, given an image.

In [39]:
# Example
test_img_path = r'test2.jpg'
img = cv2.imread(test_img_path)
features = extract_features(img, 'raw')  # be careful of the choice of feature set

In [40]:
nn = classifiers['NN']
nn.predict_proba([features])

array([[4.16532589e-11, 9.99485348e-01, 2.32585509e-05, 1.82475575e-09,
        8.27621804e-06, 3.35619836e-09, 4.83111831e-04, 4.37638396e-11,
        8.61847030e-12]])

Try to get a better accuracy by changing the model hyperparameters and retraining.